# 10. Missing values: what the theory doesn't tell you

In this class we ask four practical questions on our data:

1. Where are the gaps, and can we just delete them?
2. Which methods **can't** go in a pipeline that predicts new people, and why? (the label trap)
3. Which imputer guesses the hidden values best, and what does that depend on?
4. What does MICE add when it fills each gap several times?

Then, in `11_deployment.ipynb`: saving the model and predicting new people from the terminal.

## 1 · Setup

`course_helpers.py` (next to this notebook) is **last week's pipeline in one file**: the same functions and the same `config.yaml`, so this notebook runs on its own and only this week's *new* code appears here. The data is in `data/`.

In [9]:
import time
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer   # noqa: F401  (makes IterativeImputer importable)
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_predict

import course_helpers
from course_helpers import *        # last week's pipeline: build_pipeline, cross_validate_pipeline, CONFIG, ...
from course_helpers import _ENCODERS, _SCALERS      # names starting with _ aren't included in *

pd.set_option("display.width", 140)
X_dev, y_dev, extras_dev, cv = load_course_data()
numeric = CONFIG["preprocessing"]["numeric_features"]
categorical = CONFIG["preprocessing"]["categorical_features"]
print(f"development set: {X_dev.shape[0]} rows  ·  numeric features: {numeric}  ·  categorical: {categorical}")

development set: 5771 rows  ·  numeric features: ['age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count']  ·  categorical: ['sex', 'age_cat', 'c_charge_degree']


## 2 · Where are the gaps, and can we just delete them?

In [10]:
features = numeric + categorical
missing = X_dev[features].isna()
table = pd.DataFrame({
    "missing": missing.sum(),
    "% missing": (100 * missing.mean()).round(1),
    "reoffended | missing": [y_dev[missing[c]].mean() if missing[c].any() else np.nan for c in features],
    "reoffended | present": [y_dev[~missing[c]].mean() for c in features],
}).query("missing > 0").sort_values("missing", ascending=False)
print(table.round(3), "\n")

complete = ~missing.any(axis=1)
print(f"deleting every row with a gap keeps {complete.sum()} of {len(X_dev)} people ({complete.mean():.0%})")
print(f"reoffending rate: all {y_dev.mean():.3f}  ·  complete rows {y_dev[complete].mean():.3f}")

                 missing  % missing  reoffended | missing  reoffended | present
priors_count         397        6.9                 0.350                 0.458
c_charge_degree      192        3.3                 0.594                 0.446
juv_fel_count        174        3.0                 0.448                 0.451
age                  120        2.1                 0.492                 0.450
sex                   84        1.5                 0.452                 0.451 

deleting every row with a gap keeps 4858 of 5771 people (84%)
reoffending rate: all 0.451  ·  complete rows 0.453


- **Deleting rows** would throw away about 1 person in 6. Here the base rate barely moves, so it wouldn't even bias much. But there is a stronger reason not to: **a deployed model can't delete a person**. Someone with a gap in their file still needs a prediction, so the pipeline must know how to fill gaps anyway.
- **Deleting columns** isn't tempting either: the column with the most gaps, `priors_count`, is the strongest predictor we have.
- For two columns the gap itself **carries information**: people with a missing `priors_count` reoffend less (35% vs 46%), people with a missing `c_charge_degree` more (59% vs 45%). Filling those gaps would erase that, so they also get a `<col>_was_missing` flag (`mnar_indicator_sources` in `config.yaml`). For the others, missing or present makes no difference: filling is enough.

## 3 · The label trap

Three of the methods from the theory classes use the **class label** to fill a gap:
- mean / mode **by class**: a gap is filled with the mean of the people with the same label;
- **random-forest proximities**: the forest that decides who is "close" is grown on the label;
- the same method on an **unseen** sample: one copy per possible class, because the label isn't known.

What happens if we fill gaps by class, on all development rows, and then cross-validate?

In [11]:
model = {"type": "xgboost", "params": CONFIG["model"]["params"]}
roc = build_scorers(["roc_auc"])

honest, _ = cross_validate_pipeline(build_pipeline(CONFIG["preprocessing"], model), X_dev, y_dev, cv, roc, "roc_auc")

X_by_class = X_dev.copy()
for column in numeric:                           # fill each gap with the mean of the people with the same label
    X_by_class[column] = X_by_class[column].fillna(X_by_class.groupby(y_dev)[column].transform("mean"))
by_class, _ = cross_validate_pipeline(build_pipeline(CONFIG["preprocessing"], model), X_by_class, y_dev, cv, roc, "roc_auc")

print(f"median, inside the folds:        ROC-AUC {honest['validation'].mean():.3f}")
print(f"mean by class, on all rows:      ROC-AUC {by_class['validation'].mean():.3f}   <- looks better, but it's leakage")
print("\nwhat a missing priors_count was filled with:")
print(X_by_class.loc[X_dev["priors_count"].isna(), "priors_count"].groupby(y_dev).first().rename("filled value (by label)").to_frame().T.round(2))

median, inside the folds:        ROC-AUC 0.724
mean by class, on all rows:      ROC-AUC 0.790   <- looks better, but it's leakage

what a missing priors_count was filled with:
two_year_recid              0     1
filled value (by label)  2.19  4.87


The filled value **is the label**: every missing `priors_count` of someone who reoffended became one number, and of someone who didn't, another. XGBoost learns "this exact value → reoffends" and scores it on validation rows whose gaps were filled with *their own* label.

Two practical consequences:
- **Inside cross-validation**, anything that learns from the label must be fitted on the training rows of each fold (the rule since week 4).
- **At prediction time there is no label at all.** Filling by class can't run on a new person. That's why the theory needs the "one copy per class" trick for random forests on unseen samples. It works, but it means imputing and predicting every new person several times.

So our pipeline only uses imputers that learn from the **features**, never from the target. They run unchanged in cross-validation and on new data.

## 4 · The imputers in our pipeline

From the theory classes to scikit-learn:

| Theory | scikit-learn | What to know in practice |
|---|---|---|
| mean / median (numeric), mode (categorical) | `SimpleImputer(strategy="mean" / "median" / "most_frequent")` | the baseline to beat; already in our pipeline |
| mean / mode **by class** | not in scikit-learn | the label trap (section 3) |
| KNN | `KNNImputer(n_neighbors=k)` | numeric only (no mode for categories); distances need scaled columns; slow on big data |
| MICE | `IterativeImputer()` | by default **one** completed dataset, where each gap is the model's best guess (no random draw); the theory's several datasets in section 6 |
| random forests | `IterativeImputer(estimator=RandomForestRegressor())` ("missForest") | the forest-learns-the-relationships idea **without** proximities and without the label; the slowest |

MICE and missForest are the **same loop** in scikit-learn (`IterativeImputer`); only the regressor that predicts each column changes:

![MICE and missForest in scikit-learn](images/mice_missforest.png)

<div style="background-color:#e8f5e9; border-left:6px solid #2e7d32; padding:10px 14px; border-radius:4px"><b>🆕 New in the pipeline: <code>_NUMERIC_IMPUTERS</code></b> &nbsp;·&nbsp; <code>src/preprocessing.py</code><br>One entry per <code>numeric_strategy</code>, like <code>_SCALERS</code> and <code>_ENCODERS</code>. Each gets the <code>imputation</code> section of the config and a seed.</div>

In [12]:
# Numeric imputers. "iterative" is MICE with one completed dataset; "random_forest" is the
# missForest idea (each column predicted from the others by a random forest). Both learn only
# from the feature columns, never from the target, so they also run on data to predict.
_NUMERIC_IMPUTERS = {
    "median": lambda cfg, seed: SimpleImputer(strategy="median"),
    "mean": lambda cfg, seed: SimpleImputer(strategy="mean"),
    "knn": lambda cfg, seed: KNNImputer(n_neighbors=cfg.get("n_neighbors", 5)),
    "iterative": lambda cfg, seed: IterativeImputer(max_iter=cfg.get("max_iter", 10), random_state=seed),
    "random_forest": lambda cfg, seed: IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=cfg.get("n_estimators", 50), min_samples_leaf=5,
                                        random_state=seed, n_jobs=-1),
        max_iter=cfg.get("max_iter", 10), random_state=seed),
}

<div style="background-color:#e3f2fd; border-left:6px solid #1565c0; padding:10px 14px; border-radius:4px"><b>🔄 Changed in the pipeline: <code>build_preprocessor()</code></b> &nbsp;·&nbsp; <code>src/preprocessing.py</code><br>The numeric columns are now <b>scaled first, then imputed</b> (KNN measures distances, so every column must be on one scale; scikit-learn's scalers ignore gaps when they learn the scale). The imputer comes from <code>numeric_strategy</code>.</div>

In [13]:
def build_preprocessor(preprocessing_config: dict) -> ColumnTransformer:
    """
    Builds the unfitted ColumnTransformer described by `preprocessing_config`: scaling and
    imputation for numeric columns (scaled first, so distance-based imputers compare columns
    on one scale; scalers ignore NaN when fitting), imputation and encoding for categorical
    ones, and the missingness flags passed through. Every encoder tolerates unseen categories.
    """
    imputation = preprocessing_config.get("imputation", {})
    scaler_factory = _SCALERS[preprocessing_config["scaler"]]
    scaler = scaler_factory() if callable(scaler_factory) else scaler_factory
    encoder = _ENCODERS[preprocessing_config["encoder"]](preprocessing_config.get("random_state"))
    numeric_strategy = imputation.get("numeric_strategy", "median")
    if numeric_strategy not in _NUMERIC_IMPUTERS:
        raise ValueError(f"Unknown numeric_strategy: {numeric_strategy}. Options: {list(_NUMERIC_IMPUTERS)}")

    numeric_pipeline = Pipeline([
        ("scale", scaler),
        ("impute", _NUMERIC_IMPUTERS[numeric_strategy](imputation, preprocessing_config.get("random_state"))),
    ])
    categorical_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy=imputation.get("categorical_strategy", "most_frequent"))),
        ("encode", encoder),
    ])
    indicator_cols = [f"{c}_was_missing" for c in preprocessing_config.get("mnar_indicator_sources", [])]

    return ColumnTransformer([
        ("numeric", numeric_pipeline, preprocessing_config["numeric_features"]),
        ("categorical", categorical_pipeline, preprocessing_config["categorical_features"]),
        ("indicators", "passthrough", indicator_cols),
    ])


course_helpers.build_preprocessor = build_preprocessor     # so build_pipeline() uses the new version

The `imputation` section of `config.yaml`:
```yaml
  random_state: 42                                  # target encoder's folds and the model-based imputers
  imputation:
    numeric_strategy: "median"                      # median | mean | knn | iterative | random_forest
    categorical_strategy: "most_frequent"
    n_neighbors: 5                                  # knn
    max_iter: 10                                    # iterative, random_forest
    n_estimators: 50                                # random_forest (trees per column)
```

## 5 · Which imputer guesses best?

We can't check a guess for a real gap. So we **make** gaps where we know the answer: hide 200 known values of `age` and of `priors_count` in the validation rows of fold 1, fit each imputer on the training rows, and measure how far each guess lands from the truth (mean absolute error, in years and in priors).

In [14]:
train_idx, val_idx = next(cv.split(X_dev, y_dev))
rng = np.random.default_rng(0)


def hide_values(X, columns, n=200):
    """Hides n known values per column; returns the data with the new gaps and the hidden truth."""
    X = X.copy()
    truth = {}
    for column in columns:
        hidden = rng.choice(X.index[X[column].notna()], n, replace=False)
        truth[column] = X.loc[hidden, column].copy()
        X.loc[hidden, column] = np.nan
    return X, truth


def guess_errors(X_train, X_val, truth):
    """Mean absolute error of each imputer's guesses for the hidden values (original units)."""
    errors = {}
    for strategy, make_imputer in _NUMERIC_IMPUTERS.items():
        steps = Pipeline([("scale", RobustScaler()), ("impute", make_imputer({}, 42))]).fit(X_train)
        filled = pd.DataFrame(steps["scale"].inverse_transform(steps.transform(X_val)),
                              index=X_val.index, columns=X_val.columns)
        errors[strategy] = {column: (filled.loc[values.index, column] - values).abs().mean()
                            for column, values in truth.items()}
    return pd.DataFrame(errors)


X_val_hidden, truth = hide_values(X_dev.iloc[val_idx][numeric], ["age", "priors_count"])     # ✏️ hide more values
print("mean absolute error of the guesses (lower is better):")
print(guess_errors(X_dev.iloc[train_idx][numeric], X_val_hidden, truth).round(2))

mean absolute error of the guesses (lower is better):
              median  mean    knn  iterative  random_forest
age             9.16  9.26  10.87       8.96           8.97
priors_count    3.08  3.47   3.74       3.26           3.06


They're all about **as good as the median**. That's not because KNN, MICE or missForest are weak: it's because of what they can see. Our numeric imputer only sees the five numeric columns, and those say little about each other (the number of juvenile charges barely tells you someone's age).

But `age_cat` says a lot about `age`, and it's never missing. It's a categorical column, so our numeric imputer never sees it. Let's let the imputers see it (as 0, 1, 2):

In [15]:
age_order = {"Less than 25": 0, "25 - 45": 1, "Greater than 45": 2}


def with_age_cat(X):
    """Adds age_cat as a number (0, 1, 2), so the imputers can use it."""
    return X.assign(age_cat=X_dev.loc[X.index, "age_cat"].map(age_order))


print("mean absolute error of the guesses, now also seeing age_cat:")
print(guess_errors(with_age_cat(X_dev.iloc[train_idx][numeric]), with_age_cat(X_val_hidden), truth).round(2))

mean absolute error of the guesses, now also seeing age_cat:
              median  mean   knn  iterative  random_forest
age             9.16  9.26  4.34       4.50           4.03
priors_count    3.08  3.47  3.98       3.16           3.10


The error on `age` **halves** for KNN, MICE and missForest; the median can't use any other column, so it doesn't move. `priors_count` stays hard: nothing we have predicts it well.

**A model-based imputer is only as good as the columns it can see.** That's the practical side of the theory's point that these methods "use more of the available information": it only helps if the information is there and the imputer is given it. (Challenge B2: let the pipeline's imputer see the categorical columns.)

## 6 · MICE as the theory describes it: several completed datasets

`IterativeImputer()` gives **one** completed dataset, where each gap is the regression's best guess. The theory's MICE draws each value at random around that guess, repeats everything to get *m* datasets, trains a model on each and **combines** their predictions. In scikit-learn that's `sample_posterior=True` plus a different seed per dataset:

In [16]:
m = 5                                                # ✏️ the number of completed datasets
person = X_dev.index[X_dev["priors_count"].isna()][0]
draws, oof = [], []
for seed in range(m):
    pipeline = build_pipeline(CONFIG["preprocessing"], model)
    pipeline.set_params(prep__numeric__impute=IterativeImputer(sample_posterior=True, random_state=seed))   # draw, don't average
    oof.append(cross_val_predict(pipeline, X_dev, y_dev, cv=cv, method="predict_proba")[:, 1])
    numeric_steps = pipeline.fit(X_dev, y_dev)["prep"].named_transformers_["numeric"]     # scale -> impute, fitted
    filled = numeric_steps["scale"].inverse_transform(numeric_steps.transform(X_dev.loc[[person], numeric]))
    draws.append(filled[0, numeric.index("priors_count")])

print(f"person {person}: priors_count is missing; the {m} completed datasets filled it with {np.round(draws, 1)}")
print(f"ROC-AUC, one completed dataset (each):  {np.round([roc_auc_score(y_dev, p) for p in oof], 3)}")
print(f"ROC-AUC, the {m} models averaged:          {roc_auc_score(y_dev, np.mean(oof, axis=0)):.3f}")

person 4429: priors_count is missing; the 5 completed datasets filled it with [2.2 8.8 1.4 4.1 3.9]
ROC-AUC, one completed dataset (each):  [0.724 0.721 0.721 0.722 0.72 ]
ROC-AUC, the 5 models averaged:          0.725


- The *m* datasets disagree about the missing value: the spread **is** the uncertainty the theory talks about. A single imputation hides it.
- Averaging the *m* models (an ensemble, as in the theory's last step) gives a score at least as good as a single one, at *m* times the cost.

(Challenge B1: multiple imputation as an option of the pipeline.)

## 7 · What this notebook adds to your pipeline

| File | Change |
|---|---|
| `src/preprocessing.py` | imports `KNNImputer`, `IterativeImputer` (+ `enable_iterative_imputer`), `RandomForestRegressor`; new `_NUMERIC_IMPUTERS`; `build_preprocessor()` scales the numeric columns first, then imputes with `numeric_strategy` |
| `config.yaml` | `imputation` options: `numeric_strategy`, `n_neighbors`, `max_iter`, `n_estimators` |
| `main.py`, `src/results.py` | the imputer is printed, recorded in `runs.csv` and in the report header |

## 8 · Takeaways
- **Deleting** isn't an option for a deployed model: a new person with a gap still needs a prediction.
- Gaps that carry information get a **flag** as well as a filled value.
- **The label trap:** methods that fill gaps with the label leak in CV and can't run on new data. Our imputers learn from the features only.
- A model-based imputer is **only as good as the columns it sees**.
- **MICE** in full = several completed datasets → several models → combined; the spread is the uncertainty.

Next: `11_deployment.ipynb`.